# 11 -- Modelos relativos al nivel (fase D)

Motivación (09): un solo producto (P1632, Huevos en PRINCIPAL, demanda de ~2.600 a ~3.500 por ventana) le quita
al modelo de `intermitente` más de la mitad de su mejora sobre el piso. El modelo predijo ~560 donde la media
móvil daba ~2.800: en esa zona de volumen casi no hay filas de entrenamiento y los árboles no pueden extrapolar
una relación de escala. Si el modelo aprende **relativo al nivel** del producto, la escala deja de ser el problema.

Variantes (mismas 14 features v1, mismos hiperparámetros, sin búsqueda — solo cambia la forma del target):

| Variante | Target que aprende el modelo | Predicción final |
|---|---|---|
| `v1` | `y` (baseline, replica 08) | `y_hat` |
| `rel_w` | `y / base`, con peso `base` en la pérdida (equivale al mismo L1 en unidades originales) | `base * r_hat` |
| `rel` | `y / base`, sin pesos (error relativo) | `base * r_hat` |
| `res_mm` | `y - media_movil` | `media_movil + res_hat` (si no aprende nada, cae al piso) |
| `mezcla50` | — | `0.5 * v1 + 0.5 * media_movil` (contraste simple, sin entrenar) |

con `base = nivel_medio_60d * H + 1`. Regla de decisión igual a 07c: se elige con los folds 2-4 y se confirma una
vez en el fold 5; se adopta solo si el IC95 de la mejora sobre `v1` en 2-4 es > 0 y el fold 5 no la contradice.

Cadena: ... -> `07_matriz_as_of` -> `08` -> `09` -> **`11_modelos_relativos`**

In [1]:
import sys
sys.path.insert(0, '.')
from common_priorizacion import DW, registrar_huella, verificar_huella, purgar_entrenamiento
import evaluar_nivel1 as ev
from exportar_modelos_nivel1 import (MODELOS_POR_RAMA, RAMA_A_CONDICION, FEATURES as F1,
                                     _entrena_lightgbm_q, _entrena_tweedie_completo, _alpha_negocio_de, SEMILLA)

import json, time, warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.linear_model import TweedieRegressor
warnings.filterwarnings('ignore', category=Warning, module='sklearn')

pd.set_option('display.width', 220)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

EXPERIMENTO = 'EDA-fix-nivel1-11-modelos-relativos'
Y, PISO, H = 'target_demanda_15d', 'pred_media_movil_15d', 15
LLAVES = ['codigo_item', 'sucursal', 'fecha_origen']
B, BOOT_SEED = 2000, 0
MODOS = ['v1', 'rel_w', 'rel', 'res_mm']

verificar_huella('huella_07.json', ['matriz_as_of.parquet'])
m = pd.read_parquet(f'{DW}/matriz_as_of.parquet')
print(m.shape, '| features:', len(F1))

/home/ddelgadillo@redcorporativa.corpoica.org.co/miniforge3/envs/etl/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/09/24 18:38:39 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /home/ddelgadillo@redcorporativa.corpoica.org.co/miniforge3/envs/etl/lib/python3.11/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Huella verificada OK contra huella_07.json.
(262429, 27) | features: 14


## 1. Entrenamiento walk-forward por variante

In [2]:
def _lgb(X, y, alpha, hp, w=None):
    mod = lgb.LGBMRegressor(objective='quantile', alpha=alpha, subsample_freq=1, random_state=SEMILLA, verbose=-1, **hp)
    mod.fit(X, y, sample_weight=w)
    return mod

def _tw(X, y, hp, w=None):
    mod = TweedieRegressor(**hp)
    mod.fit(X, y, sample_weight=w)
    return mod

def predecir(rama, modo, tr, te, cols=F1):
    spec = MODELOS_POR_RAMA[rama]
    base_tr, base_te = tr['nivel_medio_60d'] * H + 1.0, te['nivel_medio_60d'] * H + 1.0
    if modo == 'v1':                                   # camino original, idéntico a 08
        X, y, Xt = tr[cols], tr[Y], te[cols]
        if spec['tipo'] == 'lightgbm':
            return _entrena_lightgbm_q(X, y, 0.5, spec['hparams']).predict(Xt)
        tw, _ = _entrena_tweedie_completo(X, y, spec['hparams_tweedie'], _alpha_negocio_de(rama))
        lg = _entrena_lightgbm_q(X, y, 0.5, spec['hparams_lgb'])
        w = spec['peso_tweedie']
        return w * np.clip(tw.predict(Xt), 0, None) + (1 - w) * lg.predict(Xt)
    if modo == 'rel_w':   y, w, inv = tr[Y] / base_tr, base_tr, (lambda p: p * base_te)
    elif modo == 'rel':   y, w, inv = tr[Y] / base_tr, None, (lambda p: p * base_te)
    elif modo == 'res_mm': y, w, inv = tr[Y] - tr[PISO], None, (lambda p: te[PISO].to_numpy() + p)
    else: raise ValueError(modo)
    X, Xt = tr[cols], te[cols]
    if spec['tipo'] == 'lightgbm':
        p = _lgb(X, y, 0.5, spec['hparams'], w).predict(Xt)
    else:
        wt = spec['peso_tweedie']
        p = wt * _tw(X, y, spec['hparams_tweedie'], w).predict(Xt) + (1 - wt) * _lgb(X, y, 0.5, spec['hparams_lgb'], w).predict(Xt)
    return np.clip(np.asarray(inv(p), dtype=float), 0, None)

partes, t0 = [], time.time()
for rama, cond in RAMA_A_CONDICION.items():
    dr = m.loc[cond(m)]
    for f in [2, 3, 4, 5]:
        tr, te = purgar_entrenamiento(dr, f, verbose=False), dr[dr['fold_id'] == f]   # sin ventanas solapadas con el test
        if len(tr) < ev.MIN_FILAS_ENTRENAMIENTO:
            print(f'{rama:22s} fold {f}: train={len(tr)} < {ev.MIN_FILAS_ENTRENAMIENTO} tras el purge -- omitida (igual que 08)'); continue
        blk = te[LLAVES + ['fold_id', Y, PISO, 'nivel_medio_60d']].copy()
        blk['rama'] = rama
        for modo in MODOS:
            blk[f'pred_{modo}'] = predecir(rama, modo, tr, te)
        blk['pred_mezcla50'] = 0.5 * blk['pred_v1'] + 0.5 * blk[PISO]
        partes.append(blk)
        print(f'{rama:22s} fold {f}: train={len(tr):>7,} test={len(te):>6,} ({time.time()-t0:,.0f}s)')
res = pd.concat(partes, ignore_index=True)
CONFIGS = MODOS + ['mezcla50']
print(res.shape)

suave_no_perecedero    fold 2: train=    260 test=   871 (4s)


suave_no_perecedero    fold 3: train=  1,129 test=   315 (9s)


suave_no_perecedero    fold 4: train=  1,448 test=   600 (14s)


suave_no_perecedero    fold 5: train=  2,051 test=   624 (18s)
suave_perecedero       fold 2: train=168 < 200 tras el purge -- omitida (igual que 08)


suave_perecedero       fold 3: train=    714 test=   200 (21s)


...

suave_perecedero       fold 5: train=  1,316 test=   444 (26s)


intermitente           fold 2: train= 20,152 test=68,198 (38s)


intermitente           fold 3: train= 88,142 test=28,850 (67s)


intermitente           fold 4: train=116,468 test=59,600 (102s)


intermitente           fold 5: train=175,878 test=76,056 (148s)
(236158, 13)


## 2. Control de replicación contra 08 (variante `v1`)

In [3]:
d08 = pd.read_parquet(f'{DW}/evaluacion_nivel_1.parquet')[LLAVES + ['rama', 'pred_q50']]
chk = res.merge(d08, on=LLAVES + ['rama'], how='inner', validate='one_to_one')
assert len(chk) == len(res)
rep = pd.DataFrame([dict(rama=r, wape_08=ev.wape(g[Y], g['pred_q50']), wape_v1=ev.wape(g[Y], g['pred_v1']),
                         dif_abs_max=float((g['pred_q50'] - g['pred_v1']).abs().max())) for r, g in chk.groupby('rama')]).set_index('rama')
print(rep)
assert (rep['wape_08'] - rep['wape_v1']).abs().max() < 0.003, 'NO reproduce 08'
print('replicación OK')

                     wape_08  wape_v1  dif_abs_max
rama                                              
intermitente           0.433    0.433        0.000
suave_no_perecedero    0.188    0.188        0.000
suave_perecedero       0.141    0.141        0.000
replicación OK


## 3. Resultados: contra el piso y contra `v1` (IC95, bootstrap de productos)

In [4]:
filas = []
for rama, g in res.groupby('rama'):
    for cfg in CONFIGS:
        col = f'pred_{cfg}'
        p = ev.mejora_con_ic(g, Y, col, PISO, B, BOOT_SEED)
        s = ev.mejora_sin_peor_producto(g, Y, col, PISO, B, BOOT_SEED)
        fila = dict(rama=rama, config=cfg, wape=p['wape_modelo'], wape_piso=p['wape_piso'], mejora_vs_piso=p['mejora_rel'],
                    piso_ic_bajo=p['ic_bajo'], piso_ic_alto=p['ic_alto'],
                    sin_peor_producto=s['producto_excluido'], mejora_vs_piso_sin_peor=s['mejora_rel'],
                    sin_peor_ic_bajo=s['ic_bajo'], sin_peor_ic_alto=s['ic_alto'])
        if cfg != 'v1':
            q = ev.mejora_con_ic(g, Y, col, 'pred_v1', B, BOOT_SEED)
            b24 = ev.mejora_con_ic(g[g['fold_id'].isin(ev.FOLDS_BUSQUEDA)], Y, col, 'pred_v1', B, BOOT_SEED)
            h5 = ev.mejora_con_ic(g[g['fold_id'] == ev.FOLD_HOLDOUT], Y, col, 'pred_v1', B, BOOT_SEED)
            fila.update(mejora_vs_v1=q['mejora_rel'], v1_ic_bajo=q['ic_bajo'], v1_ic_alto=q['ic_alto'],
                        mejora_vs_v1_f2a4=b24['mejora_rel'], f2a4_ic_bajo=b24['ic_bajo'], f2a4_ic_alto=b24['ic_alto'],
                        mejora_vs_v1_f5=h5['mejora_rel'], f5_ic_bajo=h5['ic_bajo'], f5_ic_alto=h5['ic_alto'])
        filas.append(fila)
tabla = pd.DataFrame(filas)
cols_v = ['config', 'wape', 'wape_piso', 'mejora_vs_piso', 'piso_ic_bajo', 'piso_ic_alto', 'mejora_vs_v1', 'v1_ic_bajo', 'v1_ic_alto']
for rama in tabla['rama'].unique():
    print(f'\n=== {rama} ===')
    print(tabla[tabla['rama'] == rama][cols_v].to_string(index=False))


=== intermitente ===
  config  wape  wape_piso  mejora_vs_piso  piso_ic_bajo  piso_ic_alto  mejora_vs_v1  v1_ic_bajo  v1_ic_alto
      v1 0.433      0.438           0.012        -0.061         0.056           NaN         NaN         NaN
   rel_w 0.422      0.438           0.036         0.008         0.054         0.024      -0.006       0.067
     rel 0.415      0.438           0.051         0.045         0.058         0.040      -0.002       0.106
  res_mm 0.418      0.438           0.046         0.042         0.050         0.035      -0.007       0.098
mezcla50 0.424      0.438           0.031         0.004         0.048         0.020      -0.009       0.061

=== suave_no_perecedero ===
  config  wape  wape_piso  mejora_vs_piso  piso_ic_bajo  piso_ic_alto  mejora_vs_v1  v1_ic_bajo  v1_ic_alto
      v1 0.188      0.190           0.010        -0.038         0.049           NaN         NaN         NaN
   rel_w 0.186      0.190           0.022        -0.018         0.070         0.013  

## 4. El caso P1632 y la sensibilidad al peor producto

Si la hipótesis es correcta, las variantes relativas deben corregir el error en la serie de P1632 en PRINCIPAL sin
empeorar el resto.

In [5]:
p = res[(res['rama'] == 'intermitente') & (res['codigo_item'] == 'P1632')]
filas = []
for suc, g in p.groupby('sucursal', observed=True):
    fila = {'sucursal': suc, 'filas': len(g), 'y_medio': g[Y].mean()}
    for cfg in ['piso'] + CONFIGS:
        col = PISO if cfg == 'piso' else f'pred_{cfg}'
        fila[cfg] = ev.wape(g[Y], g[col])
    filas.append(fila)
print('WAPE de P1632 por sucursal (todas las filas de test, folds 2-5)')
print(pd.DataFrame(filas).to_string(index=False))
print('\nMejora vs piso SIN el producto que más pierde, por variante:')
print(tabla[['rama', 'config', 'sin_peor_producto', 'mejora_vs_piso_sin_peor', 'sin_peor_ic_bajo', 'sin_peor_ic_alto']].to_string(index=False))

WAPE de P1632 por sucursal (todas las filas de test, folds 2-5)
 sucursal  filas   y_medio  piso    v1  rel_w   rel  res_mm  mezcla50
PRINCIPAL     40 2,861.600 0.190 0.419  0.264 0.176   0.190     0.270

Mejora vs piso SIN el producto que más pierde, por variante:
               rama   config sin_peor_producto  mejora_vs_piso_sin_peor  sin_peor_ic_bajo  sin_peor_ic_alto
       intermitente       v1             P1632                    0.049             0.040             0.057
       intermitente    rel_w             P1632                    0.049             0.042             0.056
       intermitente      rel              P608                    0.052             0.046             0.058
       intermitente   res_mm             P1449                    0.046             0.042             0.051
       intermitente mezcla50             P1632                    0.045             0.042             0.048
suave_no_perecedero       v1              P488                    0.017            -0.

## 5. Decisión mecánica

In [6]:
decision = {'notebook': '11_modelos_relativos', 'fecha_generacion': pd.Timestamp.now().isoformat(),
            'politica_holdout': ev.POLITICA_HOLDOUT, 'hiperparametros': 'fijos (MODELOS_POR_RAMA, sin busqueda)',
            'features': F1, 'replicacion_08': rep.round(4).to_dict('index'), 'ramas': {}}
for rama in tabla['rama'].unique():
    t = tabla[(tabla['rama'] == rama) & (tabla['config'] != 'v1')]
    mejor = t.loc[t['mejora_vs_v1_f2a4'].idxmax()]
    adopta = bool(mejor['f2a4_ic_bajo'] > 0 and mejor['mejora_vs_v1_f5'] > 0)
    v1 = tabla[(tabla['rama'] == rama) & (tabla['config'] == 'v1')].iloc[0]
    decision['ramas'][rama] = dict(
        mejor_config=mejor['config'], mejora_vs_v1_f2a4=float(mejor['mejora_vs_v1_f2a4']),
        ic_f2a4=[float(mejor['f2a4_ic_bajo']), float(mejor['f2a4_ic_alto'])],
        mejora_vs_v1_f5=float(mejor['mejora_vs_v1_f5']), ic_f5=[float(mejor['f5_ic_bajo']), float(mejor['f5_ic_alto'])],
        mejora_vs_piso_config=float(mejor['mejora_vs_piso']), ic_vs_piso_config=[float(mejor['piso_ic_bajo']), float(mejor['piso_ic_alto'])],
        mejora_vs_piso_v1=float(v1['mejora_vs_piso']), adopta=adopta,
        supera_piso_con_ic=bool(mejor['piso_ic_bajo'] > 0))
    print(f"{rama}: mejor {mejor['config']} | vs v1 folds 2-4 {mejor['mejora_vs_v1_f2a4']:+.1%} "
          f"IC [{mejor['f2a4_ic_bajo']:+.1%}, {mejor['f2a4_ic_alto']:+.1%}] | fold 5 {mejor['mejora_vs_v1_f5']:+.1%} | "
          f"vs piso {mejor['mejora_vs_piso']:+.1%} IC [{mejor['piso_ic_bajo']:+.1%}, {mejor['piso_ic_alto']:+.1%}] -> "
          + ('ADOPTA' if adopta else 'NO adopta (queda v1)'))
with open(f'{DW}/decision_11.json', 'w') as f:
    json.dump(decision, f, indent=2, ensure_ascii=False)
tabla.to_csv(f'{DW}/resultados_modelos_relativos_11.csv', index=False)
res.to_parquet(f'{DW}/preds_modelos_relativos_11.parquet', index=False)
huella = {'notebook': '11_modelos_relativos', 'fecha_generacion': pd.Timestamp.now().isoformat(),
          'entradas': registrar_huella(['matriz_as_of.parquet', 'evaluacion_nivel_1.parquet']),
          'salidas': registrar_huella(['decision_11.json', 'resultados_modelos_relativos_11.csv', 'preds_modelos_relativos_11.parquet'])}
with open(f'{DW}/huella_11.json', 'w') as f:
    json.dump(huella, f, indent=2, ensure_ascii=False)
print('guardado: decision_11.json, resultados_modelos_relativos_11.csv, preds_modelos_relativos_11.parquet, huella_11.json')

intermitente: mejor rel | vs v1 folds 2-4 +5.3% IC [-0.1%, +13.5%] | fold 5 +0.9% | vs piso +5.1% IC [+4.5%, +5.8%] -> NO adopta (queda v1)
suave_no_perecedero: mejor mezcla50 | vs v1 folds 2-4 +5.5% IC [+2.9%, +8.2%] | fold 5 -5.0% | vs piso +4.1% IC [+1.5%, +6.1%] -> NO adopta (queda v1)
suave_perecedero: mejor mezcla50 | vs v1 folds 2-4 +5.1% IC [+2.8%, +12.2%] | fold 5 +0.9% | vs piso +1.2% IC [-5.7%, +2.7%] -> ADOPTA


guardado: decision_11.json, resultados_modelos_relativos_11.csv, preds_modelos_relativos_11.parquet, huella_11.json


## 6. Registro en MLflow

In [7]:
for _, r in tabla.iterrows():
    metrics = {k: r[k] for k in ['wape', 'wape_piso', 'mejora_vs_piso', 'piso_ic_bajo', 'piso_ic_alto',
                                 'mejora_vs_piso_sin_peor', 'sin_peor_ic_bajo', 'sin_peor_ic_alto']}
    if r['config'] != 'v1':
        metrics.update({k: r[k] for k in ['mejora_vs_v1', 'v1_ic_bajo', 'v1_ic_alto', 'mejora_vs_v1_f2a4', 'f2a4_ic_bajo',
                                          'f2a4_ic_alto', 'mejora_vs_v1_f5', 'f5_ic_bajo', 'f5_ic_alto']})
    dec = decision['ramas'][r['rama']]
    ev.registrar_mlflow(EXPERIMENTO, f"{r['rama']}__{r['config']}",
        tags={'fase': 'D', 'notebook': '11', 'rama': r['rama'], 'config': r['config'], 'rama_git': 'fix/eda-modelado-nivel1',
              'adopta': str(dec['adopta'] and dec['mejor_config'] == r['config'])},
        params={'features': ','.join(F1), 'hiperparametros': 'fijos MODELOS_POR_RAMA', 'bootstrap_B': B,
                'base': 'nivel_medio_60d*H+1', 'huella_matriz': huella['entradas']['matriz_as_of.parquet']},
        metrics=metrics, artefactos=[f'{DW}/decision_11.json'] if r['config'] == 'mezcla50' else ())
print(f'{len(tabla)} runs registrados en {EXPERIMENTO}')

🏃 View run intermitente__v1 at: http://127.0.0.1:5000/#/experiments/17/runs/a842a234975445759dc054d9de415b06
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/17


🏃 View run intermitente__rel_w at: http://127.0.0.1:5000/#/experiments/17/runs/f901731129f742fb85995f59fb4243e0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/17


🏃 View run intermitente__rel at: http://127.0.0.1:5000/#/experiments/17/runs/57a92d8623ab4d939c6e4f02348f51da
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/17


🏃 View run intermitente__res_mm at: http://127.0.0.1:5000/#/experiments/17/runs/49c0de58921846b4915c01dbc8e571e8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/17


🏃 View run intermitente__mezcla50 at: http://127.0.0.1:5000/#/experiments/17/runs/6183a4714b044d06b5ad4f1c5df6904b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/17


...

🏃 View run suave_perecedero__v1 at: http://127.0.0.1:5000/#/experiments/17/runs/50fd5034108e455199fc0e816b20613f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/17


🏃 View run suave_perecedero__rel_w at: http://127.0.0.1:5000/#/experiments/17/runs/7492ab6b9cbb4f8bac1713c2beedd60c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/17


🏃 View run suave_perecedero__rel at: http://127.0.0.1:5000/#/experiments/17/runs/e117e09d530748de9fe6daec20b781ae
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/17


🏃 View run suave_perecedero__res_mm at: http://127.0.0.1:5000/#/experiments/17/runs/a2b76a6f94cb44309a90cdd48c93669a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/17


🏃 View run suave_perecedero__mezcla50 at: http://127.0.0.1:5000/#/experiments/17/runs/8668131c11d74ca9b282f1b353cf9890
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/17
15 runs registrados en EDA-fix-nivel1-11-modelos-relativos
